# Step 0.2 — Mapping DICOMs to the INbreast XLS metadata

This is the metadata-mapping stage of the project. Every later stage — preprocessing, patch sampling, wavelet scattering features, CNN training, fusion, and mass classification — needs to know what each DICOM is. Specifically, for each `.dcm` file we need at least:

- a clean **`file_id`** (the leading number in the filename, which is also the key in `INbreast.xls` and the stem of the OsiriX XML files),
- **`ACR`** (1–4) and the binary **`tissue_binary`** density label (ACR 1–2 → 0 non-dense, ACR 3–4 → 1 dense — the tissue-classification target),
- **`Bi-Rads`** (and `Mass`, `Micros`, `Distortion`, `Asymmetry` columns) for the mass classification task,
- **`Laterality`** and **`View`** so that Step 1.1 knows when to apply pectoral-muscle masking (only MLO/ML views) and Step 1.3 knows the orientation for patch sampling.

All of that information lives in `INbreast.xls`, and the DICOMs are stored as `<file_id>_<patient_hash>_MG_<lat>_<view>_ANON.dcm` in `data/raw/inbreast/ALL-IMGS/`. This notebook is the join between the two.

**Why this notebook matters.** If the join is wrong — say a DICOM is silently matched to the wrong row, or some DICOMs are missed — every downstream label could be wrong, but the experiments would still run and produce plausible-looking accuracy numbers. So the checks in this notebook (unmatched count, duplicate count, sheet-shape audit) are the safety net for everything that follows.

**Output.** A single CSV at `data/outputs/inbreast_index.csv` (one row per DICOM, with the columns above and the absolute path to the file). Steps 0.3, 1.1, 1.2, 1.3, 2.1, 2.2, 2.3 and 2.4 all read from this file.


## 1. Setup 

Set the project-relative path to the INbreast root folder, import the libraries we'll need, and do a one-time install of `xlrd` so pandas can read the legacy `.xls` metadata file. Notebooks run from inside `Step 0 - data_inspection NOTEBOOKS/`, so `../data/raw/inbreast` resolves to the project's `data/raw/inbreast` regardless of where the project lives on the machine.


In [1]:
from pathlib import Path
import pandas as pd

INBREAST_ROOT = Path("../data/raw/inbreast")  # adjust if needed

print("INBREAST_ROOT:", INBREAST_ROOT.resolve())
print("Exists?", INBREAST_ROOT.exists())


INBREAST_ROOT: /home/nabeel/project34/Project34/data/raw/inbreast
Exists? True


In [2]:
# pandas needs xlrd to read legacy .xls files.
# If this errors, run `pip install pandas xlrd` in your terminal instead.
%pip install -q pandas xlrd


Note: you may need to restart the kernel to use updated packages.


## 2. Initial look at the raw INbreast folder

Before opening any specific file, I audit what's actually in `data/raw/inbreast/`: how many files, what their extensions are, and whether there are any annotation / label files outside the DICOM set. The DICOMs themselves don't carry the per-image labels — those live in `INbreast.xls`. (The OsiriX XML lesion annotations live in a separate folder, `data/raw/kaggle_inbreast/AllXML/`, and are used by Steps 0.3 and 1.2 rather than here.) If the folder isn't laid out the way the pipeline expects, this is where it would show up.


In [3]:
# Audit the INbreast root: file-extension breakdown + DICOM count + label / annotation files.
from collections import Counter

# --- file-extension breakdown ---
all_files = [p for p in INBREAST_ROOT.rglob("*") if p.is_file()]
ext_counts = Counter([p.suffix.lower() if p.suffix else "<no_ext>" for p in all_files])

print("Total files:", len(all_files))
print("\nFile type counts (top):")
for ext, cnt in ext_counts.most_common(20):
    print(f"  {ext:10s}  {cnt}")

# --- DICOM count in ALL-IMGS ---
img_dir = INBREAST_ROOT / "ALL-IMGS"
if img_dir.exists():
    dcm_count = len(list(img_dir.rglob("*.dcm")))
    print(f"\nALL-IMGS found. *.dcm count: {dcm_count}")
else:
    print("\nALL-IMGS folder not found under INBREAST_ROOT.")

# --- label / annotation files anywhere under the root ---
patterns = ("*.xls", "*.xlsx", "*.csv", "*.xml", "*.json", "*.txt")
label_files = []
for pat in patterns:
    label_files.extend(INBREAST_ROOT.rglob(pat))
label_files = sorted(set(label_files))

print("\nPotential label/annotation files found:")
if not label_files:
    print("  (none found under this root)")
else:
    for p in label_files:
        print(" ", p.relative_to(INBREAST_ROOT))


Total files: 412

File type counts (top):
  .dcm        410
  .xls        1
  .ini        1

ALL-IMGS found. *.dcm count: 410

Potential label/annotation files found:
  INbreast.xls


## 3. Loading the INbreast.xls metadata

With the file inventory done, I open `INbreast.xls` and inspect what's actually in it before deciding which sheet to use.


### 3.1 Sheets, shape and columns

INbreast.xls has two sheets. The next cell prints the column list and the first 10 rows of each so I can see what the file actually looks like before deciding which sheet to use.


In [4]:
# Open each sheet and print its shape + columns + head, so the structure is visible.
xls_files = [p for p in label_files if p.suffix.lower() in (".xls", ".xlsx")]
if not xls_files:
    print("No .xls/.xlsx files found to inspect.")
else:
    xls_path = xls_files[0]
    print("Inspecting:", xls_path.relative_to(INBREAST_ROOT))

    # Choose engine based on extension
    engine = "xlrd" if xls_path.suffix.lower() == ".xls" else "openpyxl"

    xls = pd.ExcelFile(xls_path, engine=engine)
    print("Sheets:", xls.sheet_names)

    # Read each sheet and print columns + head
    for sheet in xls.sheet_names:
        df = pd.read_excel(xls_path, sheet_name=sheet, engine=engine)
        print(f"\n--- Sheet: {sheet} ---")
        print("Shape:", df.shape)
        print("Columns:", list(df.columns))
        display(df.head(10))


Inspecting: INbreast.xls
Sheets: ['Sheet1', 'Sheet2']



--- Sheet: Sheet1 ---
Shape: (412, 17)
Columns: ['Patient ID', 'Patient age', 'Laterality', 'View', 'Acquisition date', 'File Name', 'ACR', 'Bi-Rads', 'Mass ', 'Micros', 'Distortion', 'Asymmetry', 'Findings Notes (in Portuguese)', 'Other Notes', 'Lesion Annotation Status', 'Pectoral Muscle Annotation', 'Other Annotations']


,Patient ID,Patient age,Laterality,View,Acquisition date,File Name,ACR,Bi-Rads,Mass,Micros,Distortion,Asymmetry,Findings Notes (in Portuguese),Other Notes,Lesion Annotation Status,Pectoral Muscle Annotation,Other Annotations
0,removed,removed,R,CC,201001.0,22678622.0,4,1,NaN,NaN,NaN,NaN,normal,NaN,No annotation (Normal),NaN,NaN
1,removed,removed,L,CC,201001.0,22678646.0,4,3,X,NaN,NaN,NaN,nódulo,NaN,NaN,NaN,NaN
2,removed,removed,R,MLO,201001.0,22678670.0,4,1,NaN,NaN,NaN,NaN,normal,NaN,No annotation (Normal),NaN,NaN
3,removed,removed,L,MLO,201001.0,22678694.0,4,3,X,NaN,NaN,NaN,nódulo,NaN,NaN,NaN,NaN
4,removed,removed,R,CC,201001.0,22614074.0,2,5,X,X,NaN,NaN,nódulo QSE + micros,NaN,NaN,NaN,NaN
5,removed,removed,L,CC,201001.0,22614097.0,2,2,X,X,NaN,NaN,nodulo + micros,NaN,NaN,NaN,NaN
6,removed,removed,R,MLO,201001.0,22614127.0,2,5,X,X,NaN,NaN,nódulo QSE + micros,NaN,NaN,NaN,NaN
7,removed,removed,L,MLO,201001.0,22614150.0,2,2,X,X,NaN,NaN,nódulo + micros,NaN,NaN,NaN,NaN
8,removed,removed,L,MLO,201001.0,50997434.0,3,2,NaN,X,NaN,NaN,micro,NaN,NaN,NaN,NaN
9,removed,removed,R,MLO,201001.0,50997461.0,3,4a,X,X,NaN,NaN,nodulo QSE + micros,NaN,NaN,NaN,NaN



--- Sheet: Sheet2 ---
Shape: (412, 5)
Columns: ['Patient ID', 'Patient age', 'Acquisition date', 'Bi-Rads', 'DG']


,Patient ID,Patient age,Acquisition date,Bi-Rads,DG
0,10455721.0,68.0,20090826.0,2,Ca Invasor
1,10455721.0,68.0,20090826.0,5,NaN
2,10455721.0,68.0,20090826.0,2,NaN
3,10455721.0,68.0,20090826.0,5,NaN
4,10904633.0,64.0,20090825.0,4c,CDIS
5,10904633.0,64.0,20090825.0,4c,NaN
6,11223500.0,64.0,20090525.0,2,NaN
7,11223500.0,64.0,20090525.0,2,NaN
8,11223500.0,64.0,20090525.0,2,NaN
9,11223500.0,64.0,20090525.0,2,NaN


### 3.2 Observations on the XLS contents

Two sheets:

- **Sheet1** — `(412, 17)`, **one row per DICOM image**. This is the sheet we use to build the index. The relevant columns for this project are:
  - `File Name` — links to the DICOM filename (after some cleaning; see Section 5).
  - `Laterality` (L / R) and `View` (CC / MLO / ML) — drive Step 1.1's pectoral-muscle decision and Step 1.3's patch orientation.
  - `ACR` (1–4) — feeds the binary tissue-density label `tissue_binary` (Section 7).
  - `Bi-Rads` (1, 2, 3, 4a, 4b, 4c, 5, 6) — Step 1.2 maps this to a benign/malignant label for mass classification.
  - `Mass`, `Micros`, `Distortion`, `Asymmetry` — radiologist's lesion-type flags. Step 0.3 audits these against the OsiriX XML ROIs.
  - `Lesion Annotation Status` — useful for confirming which images were annotated. We carry it through into the saved index but don't act on it here.
- **Sheet2** — `(412, 5)`, **one row per patient** (`Patient ID`, `Patient age`, `Acquisition date`, `Bi-Rads`, `DG`). The `DG` column (diagnosis text) is interesting but per-patient rather than per-image, so it doesn't fit cleanly into the per-image index we're building. We don't use Sheet2 in this project.

The 412 row count is two higher than the 410 DICOMs because the XLS has two trailing blank rows; Section 5 drops them.


## 4. Listing DICOMs and extracting `file_id`

Now I look at the DICOMs themselves. The filenames follow a strict INbreast convention:

```
<file_id>_<patient_hash>_MG_<laterality>_<view>_ANON.dcm
```

The leading numeric `file_id` is the join key with the XLS `File Name` column. The cell below collects every `.dcm` path under `ALL-IMGS/` so the next sub-section can build the `file_id → path` lookup.


In [5]:
# Collect every DICOM path under ALL-IMGS.
import re
from pathlib import Path
import pandas as pd

IMG_DIR = INBREAST_ROOT / "ALL-IMGS"
dicom_paths = sorted(IMG_DIR.rglob("*.dcm"))

print("IMG_DIR:", IMG_DIR.resolve())
print("DICOM files:", len(dicom_paths))
print("Example filename:", dicom_paths[0].name if dicom_paths else "NONE")

IMG_DIR: /home/nabeel/project34/Project34/data/raw/inbreast/ALL-IMGS
DICOM files: 410
Example filename: 20586908_6c613a14b80a8591_MG_R_CC_ANON.dcm


### 4.1 `file_id` → DICOM path lookup

A regex on the filename extracts the leading digits as a string `file_id`. I keep this as a string (not an int) because the same `file_id` shows up in the XLS as e.g. `22678622.0` (float) and in the XML filenames as `22678622` (string), and the cleanest common form is the string version. The cell prints how many DICOMs gave a clean `file_id` (should be 410 / 410) and warns if any didn't.


In [6]:
def leading_id_from_filename(p: Path):
    m = re.match(r"^(\d+)", p.name)
    return m.group(1) if m else None

id_to_path = {}
bad = 0
for p in dicom_paths:
    k = leading_id_from_filename(p)
    if k is None:
        bad += 1
        continue
    # if duplicates happen, keep the first
    if k not in id_to_path:
        id_to_path[k] = p

print("IDs extracted:", len(id_to_path))
print("Files without leading numeric ID:", bad)

# show a few
for k in list(id_to_path.keys())[:5]:
    print(k, "->", id_to_path[k].name)

IDs extracted: 410
Files without leading numeric ID: 0
20586908 -> 20586908_6c613a14b80a8591_MG_R_CC_ANON.dcm
20586934 -> 20586934_6c613a14b80a8591_MG_L_CC_ANON.dcm
20586960 -> 20586960_6c613a14b80a8591_MG_R_ML_ANON.dcm
20586986 -> 20586986_6c613a14b80a8591_MG_L_ML_ANON.dcm
20587054 -> 20587054_b6a4f750c6df4f90_MG_R_CC_ANON.dcm


## 5. Joining the DICOMs to the metadata

### 5.1 Cleaning the XLS `File Name` column

The XLS stores `File Name` as a float (e.g. `22678622.0`) because pandas auto-infers the column type and there's at least one non-numeric thing in the column that prevents a clean int conversion. I convert each cell to the same string form as the regex above (`"22678622"`) so the join on `file_id` is a simple dict lookup. Rows with a missing `File Name` get `pd.NA` and are dropped in the next cell.


In [7]:
sheet1 = pd.read_excel(xls_path, sheet_name="Sheet1", engine=engine).copy()

def clean_sheet_file_id(x):
    if pd.isna(x):
        return pd.NA
    return str(int(float(x)))  # 22678622.0 -> "22678622"

sheet1["file_id"] = sheet1["File Name"].apply(clean_sheet_file_id)

print("Sheet1 rows:", len(sheet1))
print("Missing file_id rows:", sheet1["file_id"].isna().sum())

sheet1.head(3)[["File Name","file_id","Laterality","View","ACR","Bi-Rads"]]

Sheet1 rows: 412
Missing file_id rows: 2


,File Name,file_id,Laterality,View,ACR,Bi-Rads
0,22678622.0,22678622,R,CC,4,1
1,22678646.0,22678646,L,CC,4,3
2,22678670.0,22678670,R,MLO,4,1


### 5.2 Handling rows with a missing `File Name`

The XLS is 412 rows but only 410 of those have a `File Name` — the trailing two appear to be blank summary rows the INESC Porto team left in the spreadsheet. I drop them so the index has exactly one row per DICOM.


In [8]:
before = len(sheet1)
sheet1 = sheet1.dropna(subset=["file_id"]).copy()
after = len(sheet1)

print(f"Dropped {before - after} rows with missing file_id")
print("Rows remaining:", len(sheet1))

Dropped 2 rows with missing file_id
Rows remaining: 410


### 5.3 Joining by `file_id`

A `dict.get`-style merge: each XLS row's `file_id` is looked up in `id_to_path`. After this every Sheet1 row has either a populated `dicom_path` or `NaN`. The next section then checks both directions — XLS rows that didn't match a DICOM, and (implicitly, via the cardinality check) DICOMs not in the XLS.


In [9]:
sheet1["dicom_path"] = sheet1["file_id"].map(id_to_path)

matched = sheet1["dicom_path"].notna().sum()
unmatched = sheet1["dicom_path"].isna().sum()

print("Matched to DICOM:", matched)
print("Unmatched:", unmatched)

# show a few unmatched (should be small, maybe 0)
sheet1.loc[sheet1["dicom_path"].isna(), ["File Name","file_id","Laterality","View","ACR","Bi-Rads"]].head(10)

Matched to DICOM: 410
Unmatched: 0


,File Name,file_id,Laterality,View,ACR,Bi-Rads


## 6. Sanity checks — unmatched and duplicate `file_id`s

These two checks are the most important thing in the notebook. If there's a duplicate `file_id` in the XLS that maps to two different rows with different ACR / BI-RADS / view, the index would silently pick one and downstream labels for that image would depend on row order. If there's an unmatched DICOM, it would be silently dropped from the index and never train.


In [10]:
dup_ids = sheet1["file_id"].duplicated(keep=False)
print("Duplicate file_id rows (real duplicates):", dup_ids.sum())

if dup_ids.any():
    display(sheet1.loc[dup_ids, ["file_id","Laterality","View","ACR","Bi-Rads"]].sort_values("file_id").head(30))

Duplicate file_id rows (real duplicates): 0


### 6.1 Mapping audit summary

What the three checks above show:

- **All 410 DICOMs got a clean `file_id`.** Zero filenames failed the regex.
- **All 410 XLS rows (after dropping 2 blanks) matched a DICOM.** Zero unmatched.
- **Zero duplicate `file_id`s in the XLS.**

The join is a clean 1-to-1 across the whole dataset. Any future change that breaks this — for example, dropping a DICOM from `ALL-IMGS/` or editing the XLS by hand — would show up here.


## 7. Building the final index — ACR density and tissue label

For the fatty-vs-fibroglandular classification task we need a binary tissue-density target. Following the Razali paper's convention, I collapse the 4-level ACR scale into 2 classes:

- **`tissue_binary == 0`** (non-dense): ACR 1 (almost entirely fatty) and ACR 2 (scattered fibroglandular).
- **`tissue_binary == 1`** (dense): ACR 3 (heterogeneously dense) and ACR 4 (extremely dense).

The cell also coerces the ACR column to numeric (in case any cell was read as a string) and drops rows where ACR isn't one of `{1, 2, 3, 4}`. After this we have the final per-image index.


In [11]:
df = sheet1[sheet1["dicom_path"].notna()].copy()

# keep one row per image if duplicates exist
df = df.drop_duplicates(subset=["file_id"], keep="first").copy()

# Clean ACR
df["ACR"] = pd.to_numeric(df["ACR"], errors="coerce")

# keep only valid ACR rows
df = df[df["ACR"].isin([1,2,3,4])].copy()

# Binary tissue label: 0 (ACR 1-2) non-dense, 1 (ACR 3-4) dense
df["tissue_binary"] = df["ACR"].map(lambda a: 1 if a >= 3 else 0)

print("Final usable rows:", len(df))
print("\nACR distribution:")
display(df["ACR"].value_counts(dropna=False).sort_index())

print("\ntissue_binary distribution (0 non-dense, 1 dense):")
display(df["tissue_binary"].value_counts(dropna=False))

Final usable rows: 409

ACR distribution:


ACR
1.0    136
2.0    146
3.0     99
4.0     28
Name: count, dtype: int64


tissue_binary distribution (0 non-dense, 1 dense):


tissue_binary
0    282
1    127
Name: count, dtype: int64

### 7.1 Label distributions

After ACR cleanup we keep **409** of the 410 DICOMs. The one row that drops out is **`file_id 53582540`** — a right-MLO image whose `ACR` cell in the XLS is literally a whitespace-only string (a data-entry artefact in the original spreadsheet) rather than a number in `{1, 2, 3, 4}`. This image is:

- BIRADS 1 (normal, no findings of concern),
- `Mass = NaN`, `Micros = NaN`, `Distortion = NaN`, `Asymmetry = NaN`,
- `Lesion Annotation Status = "No annotation (Normal)"`,
- has no OsiriX XML file at all (it's one of the 67 DICOMs without any annotation),
- is not in the mass manifest.

Dropping it from the binary tissue task is therefore harmless — there is no mass to extract, no lesion ROI, and no downstream consumer that depends on it. The class breakdown for the remaining 409 images is:

| ACR | n | tissue_binary |
|-----|----|-----|
| 1 | 136 | 0 (non-dense) |
| 2 | 146 | 0 (non-dense) |
| 3 | 99  | 1 (dense) |
| 4 | 28  | 1 (dense) |

So the binary task is **~69 % non-dense vs. ~31 % dense** (282 vs 127). That's a moderate class imbalance, in line with what the Razali paper reports for INbreast. Downstream classifiers (Steps 2.1–2.4) do not apply class-weight balancing — performance is reported as 10-fold CV accuracy on this distribution.


## 8. Saving the index file

Write the per-image index to `data/outputs/inbreast_index.csv`. The file has one row per DICOM and is read by:

- **Step 0.3** — joins the index to the OsiriX XML lesion annotations and the two pectoral-mask sources for the annotation-source audit.
- **Step 1.1** — uses `dicom_path`, `Laterality` and `View` to decide whether to apply the pectoral-muscle mask (only MLO/ML views).
- **Step 1.2** — uses `Bi-Rads` to derive the benign/malignant label for the mass classification task (BIRADS 2–3 → benign, 4–6 → malignant).
- **Step 1.3** — uses `ACR` / `tissue_binary` to ACR-guide background-patch sampling.
- **Steps 2.1, 2.2, 2.3, 2.4** — pull `tissue_binary` as the y-label for the fatty-vs-fibroglandular classification task.


In [12]:
OUT_DIR = Path("../data/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

out = df[[
    "file_id", "dicom_path",
    "Patient ID", "Patient age",
    "Laterality", "View", "Acquisition date",
    "ACR", "tissue_binary",
    "Bi-Rads",
    "Mass ", "Micros", "Distortion", "Asymmetry",
    "Lesion Annotation Status"
]].copy()

out["dicom_path"] = out["dicom_path"].astype(str)

out_path = OUT_DIR / "inbreast_index.csv"
out.to_csv(out_path, index=False)

print("Saved:", out_path.resolve())
out.head(5)

Saved: /home/nabeel/project34/Project34/data/outputs/inbreast_index.csv


,file_id,dicom_path,Patient ID,Patient age,Laterality,View,Acquisition date,ACR,tissue_binary,Bi-Rads,Mass,Micros,Distortion,Asymmetry,Lesion Annotation Status
0,22678622,../data/raw/inbreast/ALL-IMGS/22678622_61b13c5...,removed,removed,R,CC,201001.0,4.0,1,1,NaN,NaN,NaN,NaN,No annotation (Normal)
1,22678646,../data/raw/inbreast/ALL-IMGS/22678646_61b13c5...,removed,removed,L,CC,201001.0,4.0,1,3,X,NaN,NaN,NaN,NaN
2,22678670,../data/raw/inbreast/ALL-IMGS/22678670_61b13c5...,removed,removed,R,MLO,201001.0,4.0,1,1,NaN,NaN,NaN,NaN,No annotation (Normal)
3,22678694,../data/raw/inbreast/ALL-IMGS/22678694_61b13c5...,removed,removed,L,MLO,201001.0,4.0,1,3,X,NaN,NaN,NaN,NaN
4,22614074,../data/raw/inbreast/ALL-IMGS/22614074_6bd24a0...,removed,removed,R,CC,201001.0,2.0,0,5,X,X,NaN,NaN,NaN


## 9. Visual sanity check

Six random mammograms with their `ACR`, `tissue_binary`, laterality and view overlaid. The point is to confirm by eye that the join is correct — a dense (ACR 3–4) breast on the screen should have `y=1`; a fatty one should have `y=0`. If the join were broken the labels would be visibly wrong.


**Reproducibility.** The random image selection below (a visual sanity check only) is seeded with the project-wide `SEED = 34`, so the same six mammograms appear on every run. No saved data product depends on this seed.

In [13]:
import numpy as np
import matplotlib.pyplot as plt
import pydicom

def load_dicom_for_view(path: Path) -> np.ndarray:
    """Read an INbreast DICOM and return a [0, 1] float32 array.

    Uses a 1st/99th-percentile clip + rescale.  See Step 0.1 for the
    justification (raw min-max is pulled by a few extreme edge pixels).
    All 410 INbreast files are MONOCHROME2 with no VOI LUT, so no
    inversion or windowing is applied.
    """
    ds = pydicom.dcmread(str(path))
    img = ds.pixel_array.astype(np.float32)

    lo, hi = np.percentile(img, (1, 99))
    img = np.clip(img, lo, hi)
    img = (img - lo) / (hi - lo + 1e-8)
    return img

SEED = 34  # project-wide seed; fixes which images the sanity-check grid shows
sample = out.sample(n=min(6, len(out)), random_state=SEED)

plt.figure(figsize=(12, 8))
for i, row in enumerate(sample.itertuples(index=False), 1):
    img = load_dicom_for_view(Path(row.dicom_path))
    plt.subplot(2, 3, i)
    plt.imshow(img, cmap="gray")
    plt.title(f"{row.file_id}\nACR={row.ACR}, y={row.tissue_binary}\n{row.Laterality} {row.View}", fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

/tmp/ipykernel_135588/2185295287.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Summary

What this notebook produces:

- A clean 1-to-1 join between the 410 INbreast DICOMs and `INbreast.xls` Sheet1.
- A per-image CSV at **`data/outputs/inbreast_index.csv`** with: `file_id`, `dicom_path`, patient/age (anonymised to "removed" in the public release), `Laterality`, `View`, `Acquisition date`, `ACR`, `tissue_binary`, `Bi-Rads`, and the four radiologist lesion-type columns (`Mass`, `Micros`, `Distortion`, `Asymmetry`) plus `Lesion Annotation Status`.
- The fatty-vs-fibroglandular target (`tissue_binary`) with a ~69 / 31 class split.

What this notebook does **not** do:

- It does not parse the OsiriX XML lesion annotations — that's Step 1.2 (for mass patches) and Step 0.3 (for the XML-vs-XLS audit).
- It does not derive a benign/malignant mass label from `Bi-Rads` — that's done in Step 1.2, where the BIRADS column gets the 2–3 → benign, 4–6 → malignant mapping and ROIs are extracted.
- It does not handle the pectoral-mask annotations — that join happens in Step 1.1.

**Reliability note.** The mapping went through cleanly with 0 unmatched, 0 duplicates and a coherent ACR distribution. The one DICOM that drops out at Section 7 (the 410 → 409 step) is `file_id 53582540`, a BIRADS-1 normal right-MLO image whose ACR field in the XLS is blank (whitespace only). Dropping it has no downstream effect: it carries no mass annotation, no lesion ROI, and no OsiriX XML file, so neither the tissue task nor the mass task uses it.
